In [1]:
## TO RUN ON THE CLOUD 

## preprocessed the data before starting the slide
## import embeddings <s
import os

# Define the URI to point to your manual process
os.environ["FIFTYONE_DATABASE_URI"] = "mongodb://localhost:44123"

import fiftyone as fo

# Verify connection
print(fo.core.odm.database.get_db_conn()) 


You are running the oldest supported major version of MongoDB. Please refer to https://deprecation.voxel51.com for deprecation notices. You can suppress this exception by setting your `database_validation` config parameter to `False`. See https://docs.voxel51.com/user_guide/config.html#configuring-a-mongodb-connection for more information
Database(MongoClient(host=['localhost:44123'], document_class=dict, tz_aware=False, connect=True, appname='fiftyone'), 'fiftyone')


In [2]:
import fiftyone as fo

## list all dataset
print(fo.list_datasets())

# Close any zombie sessions that might be hanging
fo.close_app()

['dugong']


In [3]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import json
from fiftyone import ViewField as F

In [2]:
# load dataset
dataset = fo.load_dataset("dugong")


In [ ]:

# ## load the views
#nc_view = dataset.match(F("region")=="NC")
#wp_view = dataset.match(F("region")=="WP")

In [4]:
dataset

Name:        dugong
Media type:  image
Num samples: 2755
Persistent:  True
Tags:        []
Sample fields:
    id:                                             fiftyone.core.fields.ObjectIdField
    filepath:                                       fiftyone.core.fields.StringField
    tags:                                           fiftyone.core.fields.ListField(fiftyone.core.fields.StringField)
    metadata:                                       fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.metadata.ImageMetadata)
    created_at:                                     fiftyone.core.fields.DateTimeField
    last_modified_at:                               fiftyone.core.fields.DateTimeField
    region:                                         fiftyone.core.fields.StringField
    subregion:                                      fiftyone.core.fields.StringField
    mission_name:                                   fiftyone.core.fields.StringField
    sea_state:                             

In [9]:
dataset.delete_sample_fields("NNN_NCtest_SEED10_raw")

In [7]:
import importlib 
import reconstruct

In [26]:
importlib.reload(reconstruct)

<module 'reconstruct' from '/share/castor/home/e2406743/code/Dugongs_IRISA-MARBEC-LIRMM/fiftyone/reconstruct.py'>

In [15]:
import sys

In [5]:
## import function 
from reconstruct import reconstruct_predictions, reconstruct_and_nms

In [6]:
def extract_stem(path):
    return Path(path).stem

extract_stem('/share/home/e2406743/code/Dugongs_IRISA-MARBEC-LIRMM/NNN_NC_SEED10_augm_0422_2254')

'NNN_NC_SEED10_augm_0422_2254'

In [ ]:
## folder of predictions
#predictions_folder = '/share/home/e2406743/code/Dugongs_IRISA-MARBEC-LIRMM/NNN_NC_SEED10_augm_0422_2254'

## folder of predictions
predictions_folder = '/share/home/e2406743/code/Dugongs_IRISA-MARBEC-LIRMM/NNN_NC_SEED10_augm_0422_2254'
name_field = extract_stem(predictions_folder)
reconstruct_predictions(
    dataset=dataset,
    predictions_dir= predictions_folder,
    field_name=name_field,
    tile_size=640
)

2026-04-25 17:05:51.263 | INFO     | reconstruct:_group_jsons_by_sample:98 - Grouped 190 JSONs → 59 unique sample stems
2026-04-25 17:05:51.266 | INFO     | reconstruct:reconstruct_predictions:280 - Matched 59 / 2755 samples to JSONs
2026-04-25 17:05:52.686 | SUCCESS  | reconstruct:reconstruct_predictions:312 - reconstruct_predictions complete → field 'NWW_partition_4_SEED10_augm_0424_1304_raw' 


In [25]:
next(iter(dataset.match(F('NWW_partition_4_SEED10_augm_0424_1304_raw').exists()).take(1)))


<SampleView: {
    'id': '69add89acfd942e1c6b5ddc8',
    'media_type': 'image',
    'filepath': '/share/home/e2406743/dataset/dataset/WP/UM/UM_M6/images/MAN_P4_UM_M6_F2_GSP_DJI_0224-658070ac304b8_281.jpeg',
    'tags': ['notin_val_0', 'notin_TEST_6', 'test_10'],
    'metadata': <ImageMetadata: {
        'size_bytes': 965674,
        'mime_type': 'image/jpeg',
        'width': 4096,
        'height': 2160,
        'num_channels': 3,
    }>,
    'created_at': datetime.datetime(2026, 3, 8, 20, 14, 18, 757000),
    'last_modified_at': datetime.datetime(2026, 4, 25, 15, 5, 52, 224000),
    'region': 'WP',
    'subregion': 'UM',
    'mission_name': 'UM_M6',
    'sea_state': 1,
    'turbidity_global': 1,
    'turbidity_local': 'Unknown',
    'sun_glitter': '25-50',
    'cloud_reflection': '0-0',
    'habitat_type': 'sand',
    'background_complexity': 'medium',
    'coral': 'P',
    'sand': 'P',
    'dense_seagrass': 'P',
    'open_sea': 'A',
    'sparse_seagrass': 'A',
    'ground_truth': <D

In [7]:
predictions_folder = '/share/home/e2406743/code/Dugongs_IRISA-MARBEC-LIRMM/output_inference/NNN_predWP_SEED10'
name_field = extract_stem(predictions_folder)
## apply non-maximumm supression
reconstruct_and_nms(
    dataset=dataset,
    predictions_dir=predictions_folder,
    field_name=name_field,
    tile_size=640
)

2026-04-26 15:06:50.517 | INFO     | reconstruct:reconstruct_and_nms:352 - Reconstructing raw detections from '/share/home/e2406743/code/Dugongs_IRISA-MARBEC-LIRMM/output_inference/NNN_predWP_SEED10'
2026-04-26 15:06:52.769 | INFO     | reconstruct:_group_jsons_by_sample:98 - Grouped 190 JSONs → 59 unique sample stems
2026-04-26 15:06:52.772 | INFO     | reconstruct:reconstruct_predictions:280 - Matched 59 / 2755 samples to JSONs
2026-04-26 15:06:54.701 | SUCCESS  | reconstruct:reconstruct_predictions:312 - reconstruct_predictions complete → field 'NNN_predWP_SEED10_raw' 
2026-04-26 15:06:54.704 | INFO     | reconstruct:reconstruct_and_nms:365 - Running NMS (iou_threshold=0.35) → field 'NNN_predWP_SEED10_nms'


 100% |███████████████████| 59/59 [1.7s elapsed, 0s remaining, 38.4 samples/s]      


2026-04-26 15:06:56.389 | SUCCESS  | reconstruct:reconstruct_and_nms:398 - reconstruct_and_nms complete → field 'NNN_predWP_SEED10_nms' saved | detections before NMS: 91 | after NMS: 70 | suppressed: 21 (23.1%)


In [8]:
dataset

Name:        dugong
Media type:  image
Num samples: 2755
Persistent:  True
Tags:        []
Sample fields:
    id:                                             fiftyone.core.fields.ObjectIdField
    filepath:                                       fiftyone.core.fields.StringField
    tags:                                           fiftyone.core.fields.ListField(fiftyone.core.fields.StringField)
    metadata:                                       fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.metadata.ImageMetadata)
    created_at:                                     fiftyone.core.fields.DateTimeField
    last_modified_at:                               fiftyone.core.fields.DateTimeField
    region:                                         fiftyone.core.fields.StringField
    subregion:                                      fiftyone.core.fields.StringField
    mission_name:                                   fiftyone.core.fields.StringField
    sea_state:                             

In [19]:
name_file.stem



'GH024197-60e713a81ab1d_1223__tile_0_540_p'

In [4]:
next(iter(dataset))

<Sample: {
    'id': '69add898cfd942e1c6b5d3ad',
    'media_type': 'image',
    'filepath': '/share/home/e2406743/dataset/dataset/NC/Flight_226/images/GH034226-619fa2d56d4d3_92.jpeg',
    'tags': ['train_0', 'val_6', 'train_10'],
    'metadata': <ImageMetadata: {
        'size_bytes': 712381,
        'mime_type': 'image/jpeg',
        'width': 2704,
        'height': 1520,
        'num_channels': 3,
    }>,
    'created_at': datetime.datetime(2026, 3, 8, 20, 14, 16, 713000),
    'last_modified_at': datetime.datetime(2026, 4, 22, 18, 8, 0, 544000),
    'region': 'NC',
    'subregion': 'NC',
    'mission_name': 'Flight_226',
    'sea_state': 0,
    'turbidity_global': 1,
    'turbidity_local': 'no',
    'sun_glitter': '0-0',
    'cloud_reflection': '0-0',
    'habitat_type': 'coral',
    'background_complexity': 'high',
    'coral': 'P',
    'sand': 'A',
    'dense_seagrass': 'A',
    'open_sea': 'P',
    'sparse_seagrass': 'A',
    'ground_truth': <Detections: {
        'detections': [


In [19]:
stem_list = [str(Path(v).stem) for v in dataset.values('filepath')]

dataset.set_values('stem_filepath',stem_list )

In [6]:
next(iter(dataset))

<Sample: {
    'id': '69add898cfd942e1c6b5d3ad',
    'media_type': 'image',
    'filepath': '/share/home/e2406743/dataset/dataset/NC/Flight_226/images/GH034226-619fa2d56d4d3_92.jpeg',
    'tags': ['train_11', 'test_37'],
    'metadata': <ImageMetadata: {
        'size_bytes': 712381,
        'mime_type': 'image/jpeg',
        'width': 2704,
        'height': 1520,
        'num_channels': 3,
    }>,
    'created_at': datetime.datetime(2026, 3, 8, 20, 14, 16, 713000),
    'last_modified_at': datetime.datetime(2026, 4, 15, 13, 34, 20, 201000),
    'region': 'NC',
    'subregion': 'NC',
    'mission_name': 'Flight_226',
    'sea_state': 0,
    'turbidity_global': 1,
    'turbidity_local': 'no',
    'sun_glitter': '0-0',
    'cloud_reflection': '0-0',
    'habitat_type': 'coral',
    'background_complexity': 'high',
    'coral': 'P',
    'sand': 'A',
    'dense_seagrass': 'A',
    'open_sea': 'P',
    'sparse_seagrass': 'A',
    'ground_truth': <Detections: {
        'detections': [
       

# old

In [40]:

import re
from pathlib import Path
from tqdm import tqdm

def get_actual_tile_metadata(x_off, y_off, img_w, img_h, tile_size=640, overlap=100):
    stride = tile_size - overlap
    x_end = min(x_off + tile_size, img_w)
    y_end = min(y_off + tile_size, img_h)
    x_start = x_end - tile_size if x_end - tile_size >= 0 else 0
    y_start = y_end - tile_size if y_end - tile_size >= 0 else 0
    tile_w = x_end - x_start
    tile_h = y_end - y_start
    return x_start, y_start, tile_w, tile_h

def reconstruct_predictions_optimized(dataset, json_folder, new_prediction_field="predictions"):
    """
    Optimized reconstruction: Groups all tile detections in memory first 
    to minimize database lookups.
    """
    # 1. Group data by stem: { stem_parent: [ (det_dict, x_off, y_off, tile_stem), ... ] }
    raw_collection = {}

    json_files = [f for f in os.listdir(json_folder) if f.endswith('.json')]
    print(f"Parsing {len(json_files)} JSON files...")

    for filename in tqdm(json_files):
        with open(os.path.join(json_folder, filename), "r") as f:
            data = json.load(f)

        tile_path = data["filepath"]
        detections = data["detections"]
        if not detections:
            continue

        # Extract offsets using your regex pattern
        match = re.search(r"__tile_(\d+)_(\d+)", tile_path)
        if not match:
            continue
        
        y_offset = int(match.group(1))
        x_offset = int(match.group(2))
        
        # Use your stem logic to identify the parent image
        tile_stem = Path(tile_path).stem
        stem_parent = str(tile_stem).split("__tile")[0]

        if stem_parent not in raw_collection:
            raw_collection[stem_parent] = []

        # Store detection info + metadata needed for math in memory
        for det in detections:
            raw_collection[stem_parent].append((det, x_offset, y_offset, tile_stem))

    # 2. Database Update Phase: Only one lookup per parent image
    print(f"Applying reconstructed detections to {len(raw_collection)} samples...")
    for stem_parent, box_entries in tqdm(raw_collection.items()):
        
        # Single database lookup for the full image based on your stem field
        sample = dataset.match(F("stem_filepath") == stem_parent).first()
        
        if not sample:
            continue
            
        img_w = sample.metadata.width
        img_h = sample.metadata.height
        final_detections = []

        # Transform all boxes associated with this image (handles 1 or 100 boxes)
        for det, x_off, y_off, t_stem in box_entries:
            # Recompute actual tile metadata
            x_start, y_start, tile_w, tile_h = get_actual_tile_metadata(
                x_off, y_off, img_w, img_h, tile_size=640, overlap=100
            )
            tx, ty, tw, th = det["bounding_box"]

            # Use actual tile size and start
            abs_x = (tx * tile_w) + x_start
            abs_y = (ty * tile_h) + y_start
            abs_w = tw * tile_w
            abs_h = th * tile_h

            # Convert to normalized coordinates
            gx = abs_x / img_w
            gy = abs_y / img_h
            gw = abs_w / img_w
            gh = abs_h / img_h

            
            final_detections.append(
                fo.Detection(
                    label=det["label"],
                    bounding_box=[gx, gy, gw, gh],
                    confidence=det["confidence"],
                    tile_source=t_stem,
                )
            )

        # Bulk save the detections to the sample prediction field
        sample[new_prediction_field] = fo.Detections(detections=final_detections)
        sample.save()

    print("Done.")



In [41]:
# --- EXECUTION ---
prediction_folder = '/share/home/e2406743/code/Dugongs_IRISA-MARBEC-LIRMM/output_inference/NNN_NC_SEED38_augm_0412_1511'
reconstruct_predictions_optimized(dataset,
                                   prediction_folder,
                                   new_prediction_field='NNN_NC_SEED38_aug')

Parsing 634 JSON files...


100%|██████████| 634/634 [00:00<00:00, 1933.93it/s]


Applying reconstructed detections to 144 samples...


100%|██████████| 144/144 [00:07<00:00, 19.54it/s]

Done.


In [25]:

# On a cluster: auto=False prevents it trying to open a browser
# Use port forwarding: ssh -L 5151:localhost:5151 user@cluster
session = fo.launch_app(dataset,
                        port=5151,
                        auto=False)
print(session.url)  

Connected to FiftyOne on port 5151 at localhost.
If you are not connecting to a remote session, you may need to start a new session and specify a port
Session launched. Run `session.show()` to open the App in a cell output.
http://localhost:5151/


In [26]:
def get_seed_from_filepath(csv_file):
    path = Path(csv_file).stem
    return path.split('_')[-1]

def return_list_from_csv(csv_file):
    dff = pd.read_csv(csv_file)
    wp_train_list = dff['train_seed'].dropna().values
    test_list = dff['test_seed'].dropna().values
    val_list = dff['val_seed'].dropna().values
    nc_train_list = dff['train_nc'].dropna().values
    return wp_train_list, nc_train_list, test_list, val_list

In [27]:
csv_file='/share/home/e2406743/dataset/df_filepaths/df_train_test_split_filepath_38.csv'
wp_train_list, nc_train_list, test_list, val_list = return_list_from_csv(csv_file)


In [30]:
np.array(test_list).tolist()

['/share/home/e2406743/dataset/dataset/WP/GAM/GAM_M10/images/MAN_P4_GAM_M10_F3_GSP_DJI_0112-66acd85262429_131.jpeg',
 '/share/home/e2406743/dataset/dataset/WP/GAM/GAM_M10/images/MAN_P4_GAM_M10_F3_GSP_DJI_0112-66acd85262429_75.jpeg',
 '/share/home/e2406743/dataset/dataset/WP/GAM/GAM_M10/images/MAN_P4_GAM_M10_F3_GSP_DJI_0112-66acd85262429_125.jpeg',
 '/share/home/e2406743/dataset/dataset/WP/GAM/GAM_M10/images/MAN_P4_GAM_M10_F3_GSP_DJI_0112-66acd85262429_71.jpeg',
 '/share/home/e2406743/dataset/dataset/WP/GAM/GAM_M1/images/MAN_P4_GAM_M1_F2_GSP_DJI_0295-64ec4698f3e31_33.jpeg',
 '/share/home/e2406743/dataset/dataset/WP/GAM/GAM_M1/images/MAN_P4_GAM_M1_F2_GSP_DJI_0309-64ec46505487e_49.jpeg',
 '/share/home/e2406743/dataset/dataset/WP/GAM/GAM_M1/images/MAN_P4_GAM_M1_F2_GSP_DJI_0299-64ec46dd2c86c_62.jpeg',
 '/share/home/e2406743/dataset/dataset/WP/GAM/GAM_M1/images/MAN_P4_GAM_M1_F2_GSP_DJI_0295-64ec4698f3e31_18.jpeg',
 '/share/home/e2406743/dataset/dataset/WP/GAM/GAM_M1/images/MAN_P4_GAM_M1_F2_G

In [32]:
# 1. Create a view that only contains these filepaths
test_view = dataset.match(F("filepath").is_in(np.array(test_list).tolist()))


# # 2. Tag the samples in this view
test_view.tag_samples("test_set_final")

# print(f"Tagged {len(test_view)} samples as 'test_set_final'")

val_view = dataset.match(F("filepath").is_in(np.array(val_list).tolist()))

# # 2. Tag the samples in this view
val_view.tag_samples("val_set_final")


In [39]:
session.view = exist_view

In [38]:
exist_view = dataset.exists("NNN_NC_SEED38_aug")

In [36]:
next(iter(dataset))

<Sample: {
    'id': '69add898cfd942e1c6b5d3ad',
    'media_type': 'image',
    'filepath': '/share/home/e2406743/dataset/dataset/NC/Flight_226/images/GH034226-619fa2d56d4d3_92.jpeg',
    'tags': ['train_11', 'test_37'],
    'metadata': <ImageMetadata: {
        'size_bytes': 712381,
        'mime_type': 'image/jpeg',
        'width': 2704,
        'height': 1520,
        'num_channels': 3,
    }>,
    'created_at': datetime.datetime(2026, 3, 8, 20, 14, 16, 713000),
    'last_modified_at': datetime.datetime(2026, 4, 15, 14, 17, 50, 976000),
    'region': 'NC',
    'subregion': 'NC',
    'mission_name': 'Flight_226',
    'sea_state': 0,
    'turbidity_global': 1,
    'turbidity_local': 'no',
    'sun_glitter': '0-0',
    'cloud_reflection': '0-0',
    'habitat_type': 'coral',
    'background_complexity': 'high',
    'coral': 'P',
    'sand': 'A',
    'dense_seagrass': 'A',
    'open_sea': 'P',
    'sparse_seagrass': 'A',
    'ground_truth': <Detections: {
        'detections': [
       

In [ ]:
def reconstruct_predictions_optimized(dataset, json_folder, new_prediction_field="predictions"):
    raw_collection = {}
    json_files = [f for f in os.listdir(json_folder) if f.endswith('.json')]
    print(f"Parsing {len(json_files)} JSON files...")

    for filename in tqdm(json_files):
        with open(os.path.join(json_folder, filename), "r") as f:
            data = json.load(f)

        tile_path = data["filepath"]
        detections = data["detections"]
        if not detections:
            continue

        # Extract tile metadata
        if "tile_metadata" not in data:
            logger.warning(f"No tile_metadata in {filename}. Skipping.")
            continue

        tile_metadata = data["tile_metadata"]
        x_start = tile_metadata["x_start"]
        y_start = tile_metadata["y_start"]
        tile_w = tile_metadata["tile_w"]
        tile_h = tile_metadata["tile_h"]

        # Use stem logic to identify the parent image
        tile_stem = Path(tile_path).stem
        stem_parent = str(tile_stem).split("__tile")[0]

        if stem_parent not in raw_collection:
            raw_collection[stem_parent] = []

        for det in detections:
            raw_collection[stem_parent].append((det, x_start, y_start, tile_w, tile_h, tile_stem))

    # Database Update Phase
    print(f"Applying reconstructed detections to {len(raw_collection)} samples...")
    for stem_parent, box_entries in tqdm(raw_collection.items()):
        sample = dataset.match(F("stem_filepath") == stem_parent).first()
        if not sample:
            continue

        img_w = sample.metadata.width
        img_h = sample.metadata.height
        final_detections = []

        for det, x_off, y_off, tile_w, tile_h, t_stem in box_entries:
            tx, ty, tw, th = det["bounding_box"]

            # Calculate global pixel coordinates using actual tile size
            abs_x = (tx * tile_w) + x_off
            abs_y = (ty * tile_h) + y_off
            abs_w = tw * tile_w
            abs_h = th * tile_h

            # Convert back to global normalized coordinates
            gx = abs_x / img_w
            gy = abs_y / img_h
            gw = abs_w / img_w
            gh = abs_h / img_h

            final_detections.append(
                fo.Detection(
                    label=det["label"],
                    bounding_box=[gx, gy, gw, gh],
                    confidence=det["confidence"],
                    tile_source=t_stem,
                )
            )

        sample[new_prediction_field] = fo.Detections(detections=final_detections)
        sample.save()

    print("Done.")